In [1]:
import logging

import pandas as pd
import torch
import torchtext

from models import *
from preprocessing import *

print("PyTorch Version : {}".format(torch.__version__))
print("Torch Text Version : {}".format(torchtext.__version__))

/home/sigma/projects/PycharmProjects/toxic-comment-detection/venv/lib/python3.11/site-packages/torchtext/data/utils.py:105: UserWarning: Spacy model "en" could not be loaded, trying "en_core_web_sm" instead
  warnings.warn(


PyTorch Version : 2.0.1+cu118
Torch Text Version : 0.15.2+cpu


In [2]:
%env WANDB_SILENT=True


logger = logging.getLogger("wandb")
logger.setLevel(logging.ERROR)

env: WANDB_SILENT=True


In [3]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda:0")

device

device(type='cuda', index=0)

In [4]:
# Load Datasets And Create Data Loaders

train_df = pd.read_csv("data/train.csv")
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
BATCH_SIZE = 256
# df_frac = 0.3
df_frac = 1.0
num_epochs = 50

## MCBiGRU (50 tokens - GlovE 300 - adam)

In [6]:
train_dl_50t_glove_300, val_dl_50t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:14, 1713.90it/s]
31915it [00:17, 1858.43it/s]


In [7]:
mcbigru = MCBiGRU(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.003,
).to(device)

mcbigru.train_model(
    train_dataloader=train_dl_50t_glove_300,
    validation_dataloader=val_dl_50t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="MCBiGRU (50 tokens - Glove_300 - adam)",
    architecture="MCBiGRU",
)

Epoch 1:
{   'Training AUROC': 0.8373019099235535,
    'Training F1_score': 0.34321314096450806,
    'Training accuracy': 0.9040650725364685,
    'Training loss': 0.44181272417366624,
    'Training precision': 0.22939243912696838,
    'Training recall': 0.6812262535095215,
    'Validation AUROC': 0.8171305060386658,
    'Validation F1_score': 0.6677643060684204,
    'Validation accuracy': 0.975763738155365,
    'Validation loss': 0.23908787178993224,
    'Validation precision': 0.6611851453781128,
    'Validation recall': 0.6744757890701294}
Epoch 2:
{   'Training AUROC': 0.9070794582366943,
    'Training F1_score': 0.6626757979393005,
    'Training accuracy': 0.9770463705062866,
    'Training loss': 0.19497406748468746,
    'Training precision': 0.7214655876159668,
    'Training recall': 0.6127452850341797,
    'Validation AUROC': 0.8745534420013428,
    'Validation F1_score': 0.6905270218849182,
    'Validation accuracy': 0.9803122878074646,
    'Validation loss': 0.13928553336858748

## MCBiGRU (25 tokens - GlovE 300 - adam)

In [6]:
train_dl_25t_glove_300, val_dl_25t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:06, 1907.23it/s]
31915it [00:15, 2106.64it/s]


In [7]:
mcbigru = MCBiGRU(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.003,
).to(device)

mcbigru.train_model(
    train_dataloader=train_dl_25t_glove_300,
    validation_dataloader=val_dl_25t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="MCBiGRU (25 tokens - Glove_300 - adam)",
    architecture="MCBiGRU",
)

Epoch 1:
{   'Training AUROC': 0.7978619933128357,
    'Training F1_score': 0.2998233437538147,
    'Training accuracy': 0.8990920782089233,
    'Training loss': 0.4500464395076813,
    'Training precision': 0.20151980221271515,
    'Training recall': 0.5853762030601501,
    'Validation AUROC': 0.8012955188751221,
    'Validation F1_score': 0.6025503873825073,
    'Validation accuracy': 0.9747714996337891,
    'Validation loss': 0.2542262909412384,
    'Validation precision': 0.6875703930854797,
    'Validation recall': 0.5362424850463867}
Epoch 2:
{   'Training AUROC': 0.8817149996757507,
    'Training F1_score': 0.6213476061820984,
    'Training accuracy': 0.9751793742179871,
    'Training loss': 0.2013742429519703,
    'Training precision': 0.7110037207603455,
    'Training recall': 0.5517705082893372,
    'Validation AUROC': 0.8546627759933472,
    'Validation F1_score': 0.6271985769271851,
    'Validation accuracy': 0.9785262942314148,
    'Validation loss': 0.14138248574733733,
 

## MCBiGRU (50 tokens - GlovE 100 - adam)

In [6]:
train_dl_50t_glove_100, val_dl_50t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:11, 1777.31it/s]
31915it [00:16, 1937.09it/s]


In [7]:
mcbigru = MCBiGRU(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.003,
).to(device)

mcbigru.train_model(
    train_dataloader=train_dl_50t_glove_100,
    validation_dataloader=val_dl_50t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="MCBiGRU (50 tokens - Glove_100 - adam)",
    architecture="MCBiGRU",
)

Epoch 1:
{   'Training AUROC': 0.7567650675773621,
    'Training F1_score': 0.23313382267951965,
    'Training accuracy': 0.8720297813415527,
    'Training loss': 0.4614175946296814,
    'Training precision': 0.14938737452030182,
    'Training recall': 0.5305722951889038,
    'Validation AUROC': 0.8379426598548889,
    'Validation F1_score': 0.5188735127449036,
    'Validation accuracy': 0.9533395767211914,
    'Validation loss': 0.2802427065372467,
    'Validation precision': 0.4169984459877014,
    'Validation recall': 0.6866182088851929}
Epoch 2:
{   'Training AUROC': 0.8484601974487305,
    'Training F1_score': 0.518449604511261,
    'Training accuracy': 0.9695862531661987,
    'Training loss': 0.2111602917403162,
    'Training precision': 0.6179166436195374,
    'Training recall': 0.44656530022621155,
    'Validation AUROC': 0.8669173717498779,
    'Validation F1_score': 0.5853461623191833,
    'Validation accuracy': 0.973194420337677,
    'Validation loss': 0.1549062136411667,
  

## MCBiGRU (25 tokens - GlovE 100 - adam)

In [6]:
train_dl_25t_glove_100, val_dl_25t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:05, 1961.84it/s]
31915it [00:14, 2128.47it/s]


In [7]:
mcbigru = MCBiGRU(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.003,
).to(device)

mcbigru.train_model(
    train_dataloader=train_dl_25t_glove_100,
    validation_dataloader=val_dl_25t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="MCBiGRU (25 tokens - Glove_100 - adam)",
    architecture="MCBiGRU",
)

Epoch 1:
{   'Training AUROC': 0.7018346190452576,
    'Training F1_score': 0.1799813061952591,
    'Training accuracy': 0.8682774901390076,
    'Training loss': 0.46945938772572304,
    'Training precision': 0.1165841817855835,
    'Training recall': 0.39451274275779724,
    'Validation AUROC': 0.6998571157455444,
    'Validation F1_score': 0.5133367776870728,
    'Validation accuracy': 0.9654133319854736,
    'Validation loss': 0.268136172413826,
    'Validation precision': 0.5311739444732666,
    'Validation recall': 0.4966586232185364}
Epoch 2:
{   'Training AUROC': 0.8340213298797607,
    'Training F1_score': 0.4834364950656891,
    'Training accuracy': 0.9690548777580261,
    'Training loss': 0.21504285516504773,
    'Training precision': 0.6224254965782166,
    'Training recall': 0.39518973231315613,
    'Validation AUROC': 0.8759783506393433,
    'Validation F1_score': 0.5639294981956482,
    'Validation accuracy': 0.9696328639984131,
    'Validation loss': 0.16483568966388704,